# Assignment 3: The "Multimodal Sentiment Engine" Challenge

**Total Marks:** 20 | **Deadline:** 7:00 PM, 22nd March, 2026 | 
**Submission:** A zip file of the folder containing this notebook, and the csv/image files you will create.


---

## Setup

Run the cell below **once** to install all required packages and download NLTK data.

In [1]:
%pip install -r requirements.txt -q

import nltk
for pkg in ["wordnet", "averaged_perceptron_tagger_eng", "punkt_tab", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
print("Setup complete!")

Note: you may need to restart the kernel to use updated packages.
Setup complete!


In [2]:
import os, re, json, time, random, warnings
from collections import Counter
from itertools import combinations

from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

warnings.filterwarnings("ignore")

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
LLM_MODEL = "google/gemini-2.0-flash-001"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sentiment-bearing words to preserve during augmentation
PRESERVE_WORDS = {
    "amazing", "terrible", "awful", "excellent", "wonderful", "horrible",
    "great", "bad", "good", "worst", "best", "love", "hate", "boring",
    "fantastic", "brilliant", "pathetic", "outstanding", "dreadful",
    "superb", "mediocre",
}

print("Imports loaded. API key present:", bool(OPENROUTER_API_KEY))

Imports loaded. API key present: True


## Task 1: Data Consolidation & Classical Augmentation (5 Marks)

**Steps:**
1. Load all three CSVs and merge them
2. Train a baseline Logistic Regression on `gold_standard_100.csv` (TF-IDF features)
3. Filter `llm_labels_150.csv` -- keep only reviews where baseline confidence ≥ 0.65 AND agrees with LLM label
4. Deduplicate by review text $\rightarrow$ save `consolidated_base.csv`
5. Identify minority class, apply 2 augmentation methods (Synonym Replacement, Back Translation)
6. Quality filter augmented samples (Jaccard similarity)
7. Save `augmented_classical.csv` and `class_distribution.png`

In [3]:
gold = pd.read_csv("gold_standard_100.csv")
weak = pd.read_csv("weak_labels_200.csv")
llm  = pd.read_csv("llm_labels_150.csv")
print(f"Gold: {len(gold)}, Weak: {len(weak)}, LLM: {len(llm)}")

def train_baseline_model(train_df, text_col="review", label_col="label"):
    """Returns (vectorizer, classifier) trained on the given dataframe."""
    vec = TfidfVectorizer(max_features=5000, stop_words="english")
    X = vec.fit_transform(train_df[text_col])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X, train_df[label_col])
    return vec, clf

vec, clf = train_baseline_model(gold)

#  1c. Filter LLM labels by confidence 
# TODO: Predict on llm reviews, keep where confidence >= 0.65 AND prediction matches LLM label
X_llm = vec.transform(llm["review"])
probs = clf.predict_proba(X_llm)
preds = clf.predict(X_llm)
llm["confidence"] = probs.max(axis=1)
llm["predicted"] = preds
filtered_llm = llm[(llm["confidence"] >= 0.65) & (llm["predicted"] == llm["label"])].copy()
filtered_llm = filtered_llm[["review", "label"]]
print("Filtered LLM samples:", len(filtered_llm))

#  1d. Merge & deduplicate 
# TODO: Combine gold + weak + filtered_llm, drop_duplicates on "review"
# Save as consolidated_base.csv
consolidated = pd.concat([gold, weak, filtered_llm], ignore_index=True)
consolidated = consolidated.drop_duplicates(subset=["review"])
print("Final dataset size:",len(consolidated))
# save dataset
consolidated.to_csv(
    "consolidated_base.csv",
    index=False
)


#  1e. Class distribution analysis 
# TODO: Count per class, identify minority, plot and save class_distribution.png
class_counts = consolidated["label"].value_counts()

print("\nClass distribution:")
print(class_counts)

# identify minority class
minority_class = class_counts.idxmin()
print("\nMinority class:", minority_class)

# plot distribution
plt.figure()

class_counts.plot(kind="bar")

plt.title("Class Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("class_distribution.png")
plt.close()

print("\nSaved files:")
print("consolidated_base.csv")
print("class_distribution.png")

Gold: 100, Weak: 220, LLM: 150
Filtered LLM samples: 27
Final dataset size: 328

Class distribution:
label
Negative    151
Neutral     115
Positive     62
Name: count, dtype: int64

Minority class: Positive

Saved files:
consolidated_base.csv
class_distribution.png


In [4]:
#  1f. Augmentation functions 

def get_wordnet_pos(tag): return wordnet.ADJ if tag.startswith("J") else wordnet.VERB if tag.startswith("V") else wordnet.NOUN if tag.startswith("N") else wordnet.ADV if tag.startswith("R") else None

def synonym_replacement(text, replace_frac=0.15):
    """Replace 15-20% of words with WordNet synonyms. Preserve sentiment-bearing words."""
    words=word_tokenize(text); tagged=pos_tag(words); new_words=words.copy(); n_replace=max(1,int(len(words)*replace_frac)); idxs=[i for i,(w,t) in enumerate(tagged) if w.lower() not in PRESERVE_WORDS]; random.shuffle(idxs); c=0
    for i in idxs:
        wn=get_wordnet_pos(tagged[i][1]); syns=wordnet.synsets(tagged[i][0],pos=wn) if wn else []; lemmas=set(l.name().replace("_"," ") for s in syns for l in s.lemmas()); lemmas.discard(tagged[i][0])
        if lemmas: new_words[i]=random.choice(list(lemmas)); c+=1
        if c>=n_replace: break
    return " ".join(new_words)

def back_translate(text, src="en", mid="hi"):
    """Translate English $\rightarrow$ Hindi $\rightarrow$ English using deep-translator GoogleTranslator."""
    from deep_translator import GoogleTranslator
    try: hi=GoogleTranslator(source=src,target=mid).translate(text); time.sleep(0.3); en=GoogleTranslator(source=mid,target=src).translate(hi); time.sleep(0.3); return en
    except Exception: return None

def quality_filter(original, augmented):
    """Return True if augmented text passes Jaccard similarity (0.30–0.95)."""
    if augmented is None: return False
    s1=set(original.lower().split()); s2=set(augmented.lower().split())
    return False if len(s1|s2)==0 else 0.30<=len(s1&s2)/len(s1|s2)<=0.95


#  1g. Apply augmentation to minority class 
# TODO: For each minority-class sample, generate 2 augmented versions (one per method)
# TODO: Apply quality_filter, keep only passing samples
# TODO: Save augmented_classical.csv

minority_df = consolidated[consolidated["label"]==minority_class]

augmented_rows=[]

for _,row in minority_df.iterrows():
    orig=row["review"]
    
    syn=synonym_replacement(orig)
    if quality_filter(orig,syn):
        augmented_rows.append({"review":syn,"label":minority_class,"method":"synonym"})
    
    bt=back_translate(orig)
    if quality_filter(orig,bt):
        augmented_rows.append({"review":bt,"label":minority_class,"method":"back_translation"})


augmented_classical=pd.DataFrame(augmented_rows)

print("Generated augmented samples:",len(augmented_classical))

augmented_classical.to_csv("augmented_classical.csv",index=False)

print("Saved: augmented_classical.csv")

Generated augmented samples: 122
Saved: augmented_classical.csv


## Task 2: LLM-Based Synthetic Review Generation (5 Marks)

**Steps:**
1. Design a few-shot prompt with 3-4 gold-standard examples
2. Use OpenRouter API (via `openai` package) to generate 300 synthetic reviews in batches of 20
3. Calculate diversity metrics: Self-BLEU per class
4. Run sentiment consistency check with baseline model $\rightarrow$ flag mismatches
5. Save `llm_generated_300.csv`, `llm_generated_flagged.csv`, `prompt_template.txt`, `diversity_report.txt`

In [5]:
from openai import OpenAI
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

# ── 2a. Few-shot prompt design ────────────────────────────────────────────────
# Pick 2 Positive, 1 Negative, 1 Neutral examples from gold standard
pos_ex = gold[gold["label"] == "Positive"].head(2)
neg_ex = gold[gold["label"] == "Negative"].head(1)
neu_ex = (gold[gold["label"] == "Neutral"].head(1)
          if "Neutral" in gold["label"].values
          else gold.iloc[[3]])

examples_json = []
for _, r in pos_ex.iterrows():
    examples_json.append({"review": r["review"], "sentiment": "Positive", "movie": "Sample Movie A"})
for _, r in neg_ex.iterrows():
    examples_json.append({"review": r["review"], "sentiment": "Negative", "movie": "Sample Movie B"})
for _, r in neu_ex.iterrows():
    examples_json.append({"review": r["review"], "sentiment": "Neutral", "movie": "Sample Movie C"})

examples_str = json.dumps(examples_json, indent=2)

PROMPT_TEMPLATE = f"""You are a movie review generator. Generate realistic, diverse movie reviews as JSON.

EXAMPLES (few-shot):
{examples_str}

INSTRUCTIONS:
- Generate exactly {{n}} movie reviews for the sentiment: {{sentiment}}
- Each review must sound natural and human-written
- Vary the writing style, length (20-80 words), vocabulary, movie references, and genres
- Use different movies for each review
- Output ONLY a valid JSON array: [{{"review": "...", "sentiment": "{{sentiment}}", "movie": "..."}}]
- Do not include any explanation or markdown — just the raw JSON array
"""

with open("prompt_template.txt", "w", encoding="utf-8") as f:
    f.write(PROMPT_TEMPLATE)
print("Saved: prompt_template.txt")

# ── 2b. Generate reviews in batches ──────────────────────────────────────────
def generate_batch(sentiment, n=20, retries=3):
    """Call OpenRouter API to generate n reviews for a given sentiment."""
    prompt = PROMPT_TEMPLATE.replace("{n}", str(n)).replace("{sentiment}", sentiment)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.9,
                max_tokens=3000,
            )
            raw = resp.choices[0].message.content.strip()
            # Extract JSON array
            match = re.search(r'\[.*\]', raw, re.DOTALL)
            if match:
                data = json.loads(match.group())
                valid = [d for d in data
                         if isinstance(d, dict) and "review" in d and "sentiment" in d]
                return valid
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(3)
    return []

# Target distribution: ~150 Positive, ~100 Negative, ~50 Neutral
targets = {"Positive": 150, "Negative": 100, "Neutral": 50}
all_reviews = []

for sentiment, total in targets.items():
    collected = []
    batch_size = 20
    n_batches = (total + batch_size - 1) // batch_size
    print(f"\nGenerating {total} {sentiment} reviews ({n_batches} batches)...")
    for i in range(n_batches):
        n = min(batch_size, total - len(collected))
        if n <= 0:
            break
        print(f"  Batch {i+1}/{n_batches} ({n} reviews)...", end=" ")
        batch = generate_batch(sentiment, n=n)
        collected.extend(batch)
        print(f"got {len(batch)}")
        time.sleep(1.5)
    collected = collected[:total]
    print(f"  Total collected: {len(collected)}")
    all_reviews.extend(collected)

# Build DataFrame
llm_generated = pd.DataFrame(all_reviews)
if "sentiment" in llm_generated.columns and "label" not in llm_generated.columns:
    llm_generated = llm_generated.rename(columns={"sentiment": "label"})
llm_generated = llm_generated[["review", "label"]].dropna()
llm_generated.to_csv("llm_generated_300.csv", index=False)
print(f"\nTotal generated: {len(llm_generated)}")
print(llm_generated["label"].value_counts())
print("Saved: llm_generated_300.csv")

# ── 2c. Self-BLEU diversity metric ───────────────────────────────────────────
def compute_self_bleu(texts, sample_size=50):
    """Lower Self-BLEU = more diverse. Target: < 0.7 per class."""
    if len(texts) < 2:
        return 0.0
    tokenized = [t.split() for t in texts[:sample_size]]
    smoothie = SmoothingFunction().method1
    scores = []
    for i, hyp in enumerate(tokenized):
        refs = [tokenized[j] for j in range(len(tokenized)) if j != i]
        score = sentence_bleu(refs, hyp, smoothing_function=smoothie)
        scores.append(score)
    return float(np.mean(scores))

print("\n=== Diversity Report ===")
diversity_lines = ["Self-BLEU Diversity Report", "=" * 40]
for sentiment in ["Positive", "Negative", "Neutral"]:
    subset = llm_generated[llm_generated["label"] == sentiment]["review"].tolist()
    bleu = compute_self_bleu(subset)
    status = "PASS" if bleu < 0.7 else "FAIL"
    line = f"{sentiment}: Self-BLEU = {bleu:.4f}  [{status} — target < 0.7]"
    print(line)
    diversity_lines.append(line)

with open("diversity_report.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(diversity_lines))
print("Saved: diversity_report.txt")

# ── 2d. Sentiment consistency check & flagging ────────────────────────────────
X_gen = vec.transform(llm_generated["review"].fillna(""))
preds = clf.predict(X_gen)
llm_generated["predicted_label"] = preds
flagged = llm_generated[llm_generated["predicted_label"] != llm_generated["label"]].copy()
flagged.to_csv("llm_generated_flagged.csv", index=False)
print(f"\nFlagged mismatches: {len(flagged)} / {len(llm_generated)}")
print("Saved: llm_generated_flagged.csv")

print("\n=== Task 2 Complete ===")

Saved: prompt_template.txt

Generating 150 Positive reviews (8 batches)...
  Batch 1/8 (20 reviews)... got 20
  Batch 2/8 (20 reviews)... got 20
  Batch 3/8 (20 reviews)... got 20
  Batch 4/8 (20 reviews)... got 20
  Batch 5/8 (20 reviews)... got 20
  Batch 6/8 (20 reviews)... got 20
  Batch 7/8 (20 reviews)... got 20
  Batch 8/8 (10 reviews)... got 10
  Total collected: 150

Generating 100 Negative reviews (5 batches)...
  Batch 1/5 (20 reviews)... got 20
  Batch 2/5 (20 reviews)... got 20
  Batch 3/5 (20 reviews)... got 20
  Batch 4/5 (20 reviews)... got 20
  Batch 5/5 (20 reviews)... got 20
  Total collected: 100

Generating 50 Neutral reviews (3 batches)...
  Batch 1/3 (20 reviews)... got 20
  Batch 2/3 (20 reviews)... got 20
  Batch 3/3 (10 reviews)... got 10
  Total collected: 50

Total generated: 300
label
Positive    150
Negative    100
Neutral      50
Name: count, dtype: int64
Saved: llm_generated_300.csv

=== Diversity Report ===
Positive: Self-BLEU = 0.1748  [PASS — target <

## Task 3: Multilingual Sentiment Translation (4 Marks)

**Steps:**
1. Sample 100 reviews (40 Pos, 40 Neg, 20 Neu), prioritize shorter reviews
2. Translate English $\rightarrow$ Hindi using `deep-translator` (`GoogleTranslator`)
3. Back-translate Hindi $\rightarrow$ English, compute BLEU score (threshold ≥ 0.3)
4. Check sentiment preservation on back-translated text
5. Manually verify 5 random samples
6. Save `bilingual_reviews.csv` with `bleu_score` and `quality_flag` columns

In [6]:
from deep_translator import GoogleTranslator
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ── 3a. Strategic sampling ────────────────────────────────────────────────────
# Reload consolidated in case kernel was fresh
consolidated = pd.read_csv("consolidated_base.csv")
consolidated["review_len"] = consolidated["review"].str.split().str.len()

sampling_targets = {"Positive": 40, "Negative": 40, "Neutral": 20}
sampled_parts = []
for label, n in sampling_targets.items():
    subset = consolidated[consolidated["label"] == label].sort_values("review_len")
    sampled_parts.append(subset.head(n))

sampled_df = pd.concat(sampled_parts).reset_index(drop=True)
print(f"Sampled {len(sampled_df)} reviews (prioritising shorter texts):")
print(sampled_df["label"].value_counts())

# ── 3b & 3c. Translation pipeline + BLEU ─────────────────────────────────────
smoothie = SmoothingFunction().method1

def translate_safe(text, source, target, max_len=4500):
    """Translate text; truncate very long texts to avoid API limits."""
    text = str(text)[:max_len]
    try:
        time.sleep(0.5)
        return GoogleTranslator(source=source, target=target).translate(text)
    except Exception as e:
        print(f"  Translation error: {e}")
        return None

rows = []
total = len(sampled_df)

for i, (_, row) in enumerate(sampled_df.iterrows()):
    orig  = row["review"]
    label = row["label"]
    print(f"[{i+1}/{total}] ({label}) {orig[:55]}...")

    # English → Hindi
    hindi = translate_safe(orig, "en", "hi")
    if hindi is None:
        hindi = orig   # fallback: keep original

    # Hindi → English (back-translation)
    back_en = translate_safe(hindi, "hi", "en")
    if back_en is None:
        back_en = orig

    # BLEU between original and back-translated
    ref  = [orig.lower().split()]
    hyp  = back_en.lower().split()
    bleu = sentence_bleu(ref, hyp, smoothing_function=smoothie)

    quality_flag = "PASS" if bleu >= 0.3 else "FAIL"

    rows.append({
        "review":         orig,
        "label":          label,
        "hindi":          hindi,
        "back_translated": back_en,
        "bleu_score":     round(bleu, 4),
        "quality_flag":   quality_flag,
    })

bilingual_df = pd.DataFrame(rows)

# ── 3d. Sentiment preservation check ─────────────────────────────────────────
X_back = vec.transform(bilingual_df["back_translated"].fillna(""))
bilingual_df["predicted_label"]    = clf.predict(X_back)
bilingual_df["sentiment_preserved"] = (
    bilingual_df["predicted_label"] == bilingual_df["label"]
)

print(f"\nQuality flags     : {bilingual_df['quality_flag'].value_counts().to_dict()}")
print(f"Sentiment preserved: "
      f"{bilingual_df['sentiment_preserved'].sum()} / {len(bilingual_df)}")

# ── 3e. Manual verification (5 random samples) ───────────────────────────────
print("\n" + "="*60)
print("Manual Verification — 5 random samples")
print("="*60)
for _, r in bilingual_df.sample(5, random_state=RANDOM_SEED).iterrows():
    print(f"\nOriginal  [{r['label']}]: {r['review']}")
    print(f"Hindi              : {r['hindi']}")
    print(f"Back-translated    : {r['back_translated']}")
    print(f"BLEU: {r['bleu_score']:.4f} | Flag: {r['quality_flag']} | "
          f"Sentiment OK: {r['sentiment_preserved']}")

# ── 3f. Save ─────────────────────────────────────────────────────────────────
bilingual_df.to_csv("bilingual_reviews.csv", index=False)
print(f"\nSaved: bilingual_reviews.csv  ({len(bilingual_df)} rows)")
print("=== Task 3 Complete ===")

Sampled 100 reviews (prioritising shorter texts):
label
Positive    40
Negative    40
Neutral     20
Name: count, dtype: int64
[1/100] (Positive) The pacing was absolutely touching. Wow, just wow....
[2/100] (Positive) The dialogue was absolutely phenomenal. Wow, just wow....
[3/100] (Positive) It perfectly balances humor and drama. Wow, just wow....
[4/100] (Positive) It perfectly balances humor and drama. Wow, simply wow....
[5/100] (Positive) The lead actor delivered a delightful performance. Wow,...
[6/100] (Positive) Wow, just wow. A must-watch for fans of the genre....
[7/100] (Positive) The lead actor delivered a heartwarming performance. A ...
[8/100] (Positive) Finally, a movie that respects the audience. Two thumbs...
[9/100] (Positive) A refreshing take on a tired genre. Highly recommended ...
[10/100] (Positive) I was completely blown away by this film. A definitive ...
[11/100] (Positive) A refreshing take on a tired genre. Don't miss this one...
[12/100] (Positive) This m

## Task 4: Multimodal Audio Generation (4 Marks)

**Steps:**
1. Select 30 reviews (10 per class) of varying lengths
2. Use `gTTS` to generate audio (`tld="com"`), convert mp3$\rightarrow$wav via `librosa`+`soundfile`
3. Extract audio features with `librosa`: duration, spectral centroid, zero crossing rate, MFCCs
4. Use `openai-whisper` (tiny model) to transcribe audio back to text, compute WER
5. Save `audio_samples/` folder, `audio_features.csv`, `audio_validation.csv`

In [7]:
from gtts import gTTS
import librosa, soundfile as sf

#  4a. Select 30 reviews (10 per class) 
# TODO: Sample from consolidated_base, mix short and long reviews

base = pd.read_csv("consolidated_base.csv")
aug = pd.read_csv("augmented_classical.csv")

combined = pd.concat([base, aug]).drop_duplicates("review")

# create length column for diversity
combined["length"] = combined["review"].str.split().str.len()

selected_samples = []

for label in ["Positive", "Negative", "Neutral"]:
    
    subset = combined[combined["label"]==label]
    
    subset = subset.sort_values("length")
    
    short = subset.head(3)
    medium = subset.iloc[len(subset)//3:len(subset)//3+4]
    long = subset.tail(3)
    
    selected = pd.concat([short, medium, long]).head(10)
    
    selected_samples.append(selected)

selected_df = pd.concat(selected_samples).reset_index(drop=True)

print("Selected samples per class:")
print(selected_df["label"].value_counts())



#  4b. TTS generation 
os.makedirs("audio_samples", exist_ok=True)

# TODO: For each review, generate audio with gTTS (tld="com")
# Save as mp3, then load with librosa and re-save as .wav via soundfile

audio_paths = []

for i,row in selected_df.iterrows():
    
    text = row["review"]
    
    label = row["label"]
    
    mp3_path = f"audio_samples/sample_{i}.mp3"
    
    wav_path = f"audio_samples/sample_{i}.wav"
    
    tts = gTTS(text=text, lang="en", tld="com")
    
    tts.save(mp3_path)
    
    y,sr = librosa.load(mp3_path, sr=None)
    
    sf.write(wav_path, y, sr)
    
    audio_paths.append(wav_path)



#  4c. Audio feature extraction 
# TODO: For each wav, extract with librosa:
#   - duration (librosa.get_duration)
#   - spectral centroid (librosa.feature.spectral_centroid)
#   - zero crossing rate (librosa.feature.zero_crossing_rate)
#   - MFCCs (librosa.feature.mfcc, n_mfcc=13, take mean)
# Save audio_features.csv

feature_rows = []

for i,path in enumerate(audio_paths):
    
    y,sr = librosa.load(path, sr=None)
    
    duration = librosa.get_duration(y=y, sr=sr)
    
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr).mean()
    
    zcr = librosa.feature.zero_crossing_rate(y).mean()
    
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13).mean(axis=1)
    
    row = {
        "file": path,
        "label": selected_df.loc[i,"label"],
        "duration": duration,
        "spectral_centroid": spectral_centroid,
        "zero_crossing_rate": zcr
    }
    
    for j,val in enumerate(mfcc):
        row[f"mfcc_{j+1}"] = val
    
    feature_rows.append(row)

audio_features = pd.DataFrame(feature_rows)

audio_features.to_csv("audio_features.csv", index=False)

print("Saved audio_features.csv")



#  4d. Whisper round-trip validation 
import whisper

_whisper_model = whisper.load_model("tiny")

# TODO: Transcribe each wav with Whisper
# Compute WER (word-level Levenshtein distance / reference word count)
# Flag samples with WER > 0.25
# Save audio_validation.csv

def word_error_rate(ref, hyp):
    
    ref_words = ref.split()
    
    hyp_words = hyp.split()
    
    d = [[0]*(len(hyp_words)+1) for _ in range(len(ref_words)+1)]
    
    for i in range(len(ref_words)+1):
        d[i][0] = i
    
    for j in range(len(hyp_words)+1):
        d[0][j] = j
    
    for i in range(1,len(ref_words)+1):
        
        for j in range(1,len(hyp_words)+1):
            
            if ref_words[i-1] == hyp_words[j-1]:
                cost = 0
            else:
                cost = 1
            
            d[i][j] = min(
                d[i-1][j] + 1,
                d[i][j-1] + 1,
                d[i-1][j-1] + cost
            )
    
    return d[len(ref_words)][len(hyp_words)] / max(1,len(ref_words))


validation_rows = []

for i,path in enumerate(audio_paths):
    
    result = _whisper_model.transcribe(path)
    
    predicted_text = result["text"]
    
    original_text = selected_df.loc[i,"review"]
    
    wer = word_error_rate(
        original_text.lower(),
        predicted_text.lower()
    )
    
    validation_rows.append({
        "file": path,
        "label": selected_df.loc[i,"label"],
        "original_text": original_text,
        "transcribed_text": predicted_text,
        "wer": wer,
        "quality_flag": wer <= 0.25
    })

audio_validation = pd.DataFrame(validation_rows)

audio_validation.to_csv("audio_validation.csv", index=False)

print("Saved audio_validation.csv")



print("\nGenerated files:")
print("audio_samples/ (30 wav files)")
print("audio_features.csv")
print("audio_validation.csv")

Selected samples per class:
label
Positive    10
Negative    10
Neutral     10
Name: count, dtype: int64
Saved audio_features.csv
Saved audio_validation.csv

Generated files:
audio_samples/ (30 wav files)
audio_features.csv
audio_validation.csv


## Task 5: Final Dataset Assembly & Model Evaluation (2 Marks)

**Steps:**
1. Merge all datasets: `consolidated_base.csv` + `augmented_classical.csv` + `llm_generated_300.csv` (excluding flagged) + English text from `bilingual_reviews.csv`
2. Deduplicate $\rightarrow$ save `final_augmented_dataset.csv`
3. Use `BlackBoxEvaluator` from `evaluator.py` to compare baseline vs augmented accuracy

In [8]:
from evaluator import BlackBoxEvaluator

# ── 5a. Assemble final dataset ────────────────────────────────────────────────
consolidated_base = pd.read_csv("consolidated_base.csv")
augmented_classical = pd.read_csv("augmented_classical.csv")
llm_generated_300 = pd.read_csv("llm_generated_300.csv")
bilingual_reviews = pd.read_csv("bilingual_reviews.csv")
llm_flagged = pd.read_csv("llm_generated_flagged.csv")

# Exclude flagged LLM reviews
flagged_texts = set(llm_flagged["review"].tolist())
llm_clean = llm_generated_300[~llm_generated_300["review"].isin(flagged_texts)].copy()
print(f"LLM reviews after removing flagged: {len(llm_clean)} "
      f"(removed {len(llm_generated_300) - len(llm_clean)})")

# Back-translated English from bilingual_reviews (PASS quality only)
back_translated = bilingual_reviews[bilingual_reviews["quality_flag"] == "PASS"][
    ["back_translated", "label"]
].rename(columns={"back_translated": "review"})
print(f"Back-translated reviews (PASS): {len(back_translated)}")

# Merge everything
final_augmented = pd.concat(
    [
        consolidated_base[["review", "label"]],
        augmented_classical[["review", "label"]],
        llm_clean[["review", "label"]],
        back_translated[["review", "label"]],
    ],
    ignore_index=True,
)
final_augmented = final_augmented.drop_duplicates(subset=["review"]).dropna()
final_augmented.to_csv("final_augmented_dataset.csv", index=False)

print(f"\nFinal augmented dataset: {len(final_augmented)} samples")
print(final_augmented["label"].value_counts())
print("Saved: final_augmented_dataset.csv")

# ── 5b. Black-Box Evaluation ──────────────────────────────────────────────────
evaluator = BlackBoxEvaluator()
test_df = pd.read_csv("gold_standard_100.csv")

# Baseline evaluation (consolidated_base only)
baseline_acc = evaluator.run_evaluation(
    consolidated_base, test_df, model_name="Baseline (consolidated_base)"
)

# Augmented evaluation (full dataset)
augmented_acc = evaluator.run_evaluation(
    final_augmented, test_df, model_name="Augmented (final_augmented_dataset)"
)

# Print comparison
print("=" * 50)
print(f"Baseline accuracy:  {baseline_acc:.2%}")
print(f"Augmented accuracy: {augmented_acc:.2%}")
print(f"Improvement:        {augmented_acc - baseline_acc:+.2%}")
print("=" * 50)
print("=== Task 5 Complete ===")

LLM reviews after removing flagged: 144 (removed 156)
Back-translated reviews (PASS): 81

Final augmented dataset: 631 samples
label
Positive    236
Negative    230
Neutral     165
Name: count, dtype: int64
Saved: final_augmented_dataset.csv
Initializing Black-Box Embedder...
Embedder loaded successfully.

--- Evaluating: Baseline (consolidated_base) ---
Training on 228 samples (excluded 100 test overlaps)...
Accuracy: 76.00%
Classification Report:
              precision    recall  f1-score   support

    Negative       0.53      0.84      0.65        25
     Neutral       0.86      0.68      0.76        47
    Positive       1.00      0.82      0.90        28

    accuracy                           0.76       100
   macro avg       0.80      0.78      0.77       100
weighted avg       0.82      0.76      0.77       100

----------------------------------------

--- Evaluating: Augmented (final_augmented_dataset) ---
Training on 531 samples (excluded 100 test overlaps)...
Accuracy: 85